# 强化学习训练环境准备

本课程是 Qwen3-1.7B 强化学习（RL）实战教程的起点。在完成前置 SFT 课程后，你将进入一个全新的训练范式——强化学习。本章将帮助你搭建 RL 训练所需的软件环境，并理解 RL 训练框架与 SFT 框架的核心差异。

---

## CANNLab 环境说明

进入 CANNLab 后，默认工作目录为 `cann-recipes-train` 仓库根目录（`/mnt/workspace/gitCode/cann/cann-recipes-train`）。本课程的训练文件（启动脚本、数据、patch 等）位于该仓库的 `llm_rl/qwen3_wordle/` 目录下。

本课程 notebook 位于另一个仓库 `cann-learning-hub`，需先在**终端**中执行以下命令克隆到与 `cann-recipes-train` 同级目录（只需执行一次，否则无法打开后续 notebook）：

```bash
cd /mnt/workspace/gitCode/cann
git clone https://gitcode.com/cann/cann-learning-hub.git
```

完成后的目录结构：

```text
/mnt/workspace/gitCode/cann/
├── cann-recipes-train/        <- CANNLab 默认工作目录
│   └── llm_rl/qwen3_wordle/   <- 训练工作目录
└── cann-learning-hub/         <- 课程 notebook（本仓库）
    └── tutorials/rl_training_pipeline/
```

> 后续所有 notebook 中的命令均假设 Jupyter 启动时工作目录为 `cann-recipes-train` 根目录，请勿手动切换。

---

## 前置要求

为了充分掌握本章内容，你应已具备以下能力：

- 已完成 Qwen3-1.7B SFT 性能调优实战，理解 SFT 训练流程。
- 了解 PyTorch 模型训练的基本流程（forward → loss → backward → update）。
- 已访问过 Ascend NPU 环境（`npu-smi info` 可用）。
- SFT 模型权重已生成（`Qwen3-1.7B-Wordle-SFT`）。

无须具备任何强化学习相关的前置知识，本章将从零开始介绍 RL 训练的基本概念。

---

## 章节目标

完成本章后，你将能够：

- 理解 RL 训练与 SFT 训练的本质区别。
- 理解 verl 框架在 RL 训练中的角色及其核心架构。
- 完成 verl + vllm-ascend 的安装与环境验证。
- 确认 SFT 模型权重可用，为后续训练做好准备。

---

## 本章内容

- [1.1 章节介绍](01.01_chapter_intro.ipynb)
- [1.2 安装 verl + vllm-ascend](01.02_install_verl_and_vllm_ascend.ipynb)
- [1.3 verl 框架架构速览](01.03_verl_framework_overview.ipynb)
- [1.4 章节练习](01.04_chapter_practice.ipynb)

---

## SFT vs RL：训练范式的转变

在 SFT（监督微调）中，模型学习的是标准答案——给定输入，模型输出的每一步都有正确参考，loss 由 cross-entropy 直接计算。

RL（强化学习）通过**试错**学习：模型生成多条回复，由奖励函数（Reward Function）打分，好的回复被强化，差的被抑制。RL 不要求逐 token 的标准输出，但 Wordle 仍需 `ground_truth` 秘密单词，才能自动判断猜词是否正确。

<img src="./images/sft_vs_rl_flow.png" alt="SFT 与 RL 训练流程对比" width="90%">

SFT 的训练目标是模仿完整的标准输出，loss 可以直接由模型输出和标签之间的差异计算。RL / GRPO 不需要逐 token 的标准输出，训练过程先生成多条 rollout，再根据秘密单词计算奖励，并通过相对优势决定哪些输出应该被强化、哪些应该被抑制。

核心区别：

- **SFT** 需要输入 + 标准答案，模型模仿答案。
- **RL** 只需要输入 + 奖励规则，模型通过试错优化策略。

这意味着 RL 训练多了一个关键组件——**推理引擎（vLLM）**：训练过程中需要不断让模型生成文本（rollout），用来计算奖励。而 verl 正是连接训练和推理的 RL 框架。

> **为什么不能直接用 SFT 训练 Wordle？**
>
> SFT 需要每一步的最优猜测作为标签，但 Wordle 的最优策略取决于游戏进行中的具体反馈，无法预先标注。RL 通过让模型自己试错、根据结果获得奖励，天然适合这种多轮决策场景。